In [1]:
from pyspark.sql import SparkSession
from delta import *

In [2]:
builder = SparkSession.builder.appName("Local_Lakehouse") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")\
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .config("spark.sql.caseSensitive", "true") \
    .master("local[4]")

In [3]:
spark = configure_spark_with_delta_pip(builder).getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/13 12:56:30 WARN Utils: Your hostname, LAPTOP-MSL0MUJB, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/13 12:56:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/harsh/wsl_venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/harsh/.ivy2.5.2/cache
The jars for the packages stored in: /home/harsh/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5103a264-d2b5-4629-ba90-6d3626e04a86;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache

In [4]:
df = spark.read \
    .format("parquet") \
    .load("/mnt/d/Bluesky-Reddit-BigDataProject/Bluesky_data/silver/vaderSentimentTable/")

In [5]:
df.show(n=100,truncate=False)

+---------------+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [6]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [7]:
df = df.filter(col("commit.record.langs") == array(lit("en")))

In [8]:
df=df.withColumn("timestamp",timestamp_micros(col("time_us")))

In [9]:
df=df.withColumn("splitted_text",split(col("commit.record.text"),r"\s+"))

In [10]:
df=df.withColumn("exploded_text",explode(col("splitted_text")))

In [11]:
df=df.groupBy(col("exploded_text").alias("word"),window(col("timestamp"),"2 hours").alias("time_range")).agg(avg(col("vader_sentiment_score")).alias("avg_vader_sentiment_score"),count("*").alias("word_count"))

26/04/13 12:56:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [12]:
df.show(n=100,truncate=False)

+------------------------------------------------+------------------------------------------+-------------------------+----------+
|word                                            |time_range                                |avg_vader_sentiment_score|word_count|
+------------------------------------------------+------------------------------------------+-------------------------+----------+
|9.26MHz.                                        |{2026-04-03 08:00:00, 2026-04-03 10:00:00}|0.12800000607967377      |14        |
|💛                                              |{2026-04-03 08:00:00, 2026-04-03 10:00:00}|0.37892916426062584      |24        |
|findingtimetowrite.wordpress.com/2026/04/03/f...|{2026-04-03 08:00:00, 2026-04-03 10:00:00}|0.0                      |1         |
|aware                                           |{2026-04-03 08:00:00, 2026-04-03 10:00:00}|0.010037877342917702     |66        |
|Blessed                                         |{2026-04-03 08:00:00, 2026-04-03 1

In [13]:
df.count()

18026744

In [14]:
df.write.format("parquet")\
        .mode("append")\
        .save("/mnt/d/Bluesky-Reddit-BigDataProject/Bluesky_data/silver/vaderSentimentAnalysisFinal/")